# Algoritmo de clasificación de imágenes

In [ ]:
# Importar paquetes
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from keras.models import load_model

from dataset_preparation import prepare_dataset
from visualization_preprocessing import create_data_generators, visualize_dataset
from model_builder import build_model
from train import evaluate_model, train_model

In [ ]:
# Preparar el conjunto de datos
raw_train_dir = Path("../data/raw/train")
dataset_dir = Path("../data/processed/dataset")

if not dataset_dir.exists() or not any(dataset_dir.iterdir()):
    prepare_dataset(raw_train_dir, dataset_dir, train_split=0.8)

# Visualizar las primeras imágenes de perros y gatos
visualize_dataset(dataset_dir)

#### Crear los conjuntos de datos de imágenes

In [ ]:
# Crear generadores para entrenamiento y validación
trdata, tsdata = create_data_generators(dataset_dir)
print(trdata.class_indices)
print(f"Imágenes de entrenamiento: {trdata.samples}")
print(f"Imágenes de prueba: {tsdata.samples}")

#### Entrenar el modelo

In [ ]:
# Construir y compilar EfficientNet-B0
model = build_model()
model.summary()

In [ ]:
# Entrenar el modelo y guardar el mejor checkpoint
checkpoint_path = Path("../models/checkpoint_best.keras")
hist = train_model(model, trdata, tsdata, checkpoint_path, epochs=20)

In [ ]:
# Evaluar el mejor modelo
best_model = load_model(checkpoint_path)
evaluate_model(best_model, tsdata)

In [ ]:
# Visualizar exactitud y pérdida
plt.plot(hist.history["accuracy"])
plt.plot(hist.history["val_accuracy"])
plt.plot(hist.history["loss"])
plt.plot(hist.history["val_loss"])
plt.title("Rendimiento del modelo")
plt.xlabel("Época")
plt.legend(["Exactitud", "Exactitud de validación", "Pérdida", "Pérdida de validación"])
plt.show()

#### Guardar y probar el modelo

In [ ]:
# Guardar el modelo entrenado
final_model_path = Path("../models/dogs_vs_cats_efficientnet.keras")
best_model.save(final_model_path)
print(f"Modelo guardado en {final_model_path}")

In [ ]:
# Predecir una imagen del conjunto de prueba
from keras.preprocessing.image import load_img, img_to_array

test_image_path = next((dataset_dir / "test" / "dogs").glob("*.jpg"), None)
if test_image_path is None:
    test_image_path = next((dataset_dir / "test" / "cats").glob("*.jpg"), None)

img = load_img(test_image_path, target_size=(224, 224))
img_array = img_to_array(img) / 255.0
output = best_model.predict(np.expand_dims(img_array, axis=0), verbose=0)
class_names = {index: name for name, index in trdata.class_indices.items()}
prediction = class_names[int(np.argmax(output[0]))]
plt.imshow(img)
plt.title(f"Predicción: {prediction}")
plt.axis("off")
plt.show()